# Poster campaign · 3/3 — Figures (plot only)

Generate the agreed **five poster plots** from one verified source run and the saved 27-configuration SI table. This notebook performs **no expensive physics or CLs calculations**; rerun it freely for formatting changes. All numeric outputs come from the repository's calculated arrays, not visually digitized LEGEND curves.

The default plot styling uses the supplied DirectDetectionPlots aesthetic (sans-serif, colored limits, dashed projections). If a local DirectDetectionPlots checkout is available, its **numeric** updated CDEX/SuperCDMS/LZ/XENONnT curves are used; otherwise we fall back to the older, provenance-frozen comparisons bundled with WIMP-LEGEND. The use of a plot comparison curve does not synchronize differing halo/threshold conventions.

In [ ]:
from pathlib import Path
import sys

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'analyses/legend_poster').is_dir():
            return candidate
    raise RuntimeError('Open this notebook in the WIMP-LEGEND checkout (or a subdirectory).')

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CAMPAIGN_DIR = ROOT / 'analyses/legend_poster/campaigns/poster_final'
SPEC_PATH = ROOT / 'analyses/legend_poster/poster_campaign_spec.json'
CONFIG_PATH = CAMPAIGN_DIR / 'config.json'
print('Repository:', ROOT)
print('Campaign:', CAMPAIGN_DIR)


## Load and verify the completed numerical campaign

In [ ]:
import json
from analyses.legend_poster.campaign_scripts.poster_campaign import load_campaign, load_spec

source_index = json.loads((CAMPAIGN_DIR/'source_run.json').read_text())
RUN_DIR = ROOT/source_index['run_relative']
rows, manifest = load_campaign(CAMPAIGN_DIR)
spec = load_spec(SPEC_PATH)
assert manifest['source_run_id'] == source_index['run_id']
print('Verified limit table:',len(rows),'rows')
print('Source run:',RUN_DIR)
print('Three backgrounds:', manifest['background_modes'])
print('Thresholds [keVee]:',manifest['threshold_keVee'])
print('Exposures [t·yr]:',manifest['exposure_tonne_year'])


## Plot settings (change freely)

The uploaded DirectDetectionPlots notebook used `PlotBound`-style comparisons. Our helper reads the same two-column published-limit files when available. The five resulting PDF figures are editable as vector artwork.

**Note:** Figure 1 displays bin-width-normalized, detector-folded counts, not an exact unbinned differential function; the working bins are 10 eVee below 2 keV. We do not interpolate across missing mass sensitivity. External projections are marked as such.

In [ ]:
import os
from pathlib import Path
from analyses.legend_poster.campaign_scripts.poster_figures import plot_five

# Point to the cloned DirectDetectionPlots repository, or leave None to use the
# frozen experimental curves already shipped with WIMP-LEGEND.
DDP_ROOT = Path(os.environ['DIRECT_DETECTION_PLOTS_ROOT']) if os.environ.get('DIRECT_DETECTION_PLOTS_ROOT') else None

MASS_XLIM = (1., 100.)
SI_YLIM = (1e-49, 1e-40)
FIGURE_DPI = 220
FIGURE_DIR = CAMPAIGN_DIR/'figures'

paths = plot_five(CAMPAIGN_DIR, RUN_DIR, SPEC_PATH, ROOT,
                  ddp_root=DDP_ROOT, xlim=MASS_XLIM,
                  ylim=SI_YLIM, dpi=FIGURE_DPI,
                  figure_dir=FIGURE_DIR)
print('Figures saved in:', paths)
for p in sorted(paths.glob('*.pdf')):
    print(' -',p.name)


## Preview figures without recalculation

In [ ]:
from IPython.display import display, Image
for name in ('01_energy_spectra','02_neutrino_best_case',
             '03_background_comparison','04_threshold_comparison',
             '05_exposure_comparison'):
    print(name)
    display(Image(filename=str(FIGURE_DIR/(name+'.png')),width=900))
